In [1]:
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
df = pd.read_csv("Income.csv")

In [3]:
#print all the coloumns in the dataset
print(df.columns)

Index(['Unnamed: 0', 'HH_ID', 'STATE', 'HR', 'DISTRICT', 'REGION_TYPE',
       'STRATUM', 'PSU_ID', 'MONTH_SLOT', 'MONTH', 'RESPONSE_STATUS',
       'NR_REASON', 'FAMILY_SHIFTED', 'R_HH_WGT_MS', 'R_HH_WGT_FOR_COUNTRY_MS',
       'R_HH_WGT_FOR_STATE_MS', 'HH_NR_MS', 'HH_NR_FOR_COUNTRY_MS',
       'HH_NR_FOR_STATE_MS', 'AGE_GROUP', 'OCCUPATION_GROUP', 'EDU_GROUP',
       'GENDER_GROUP', 'SIZE_GROUP', 'TOT_INC', 'INC_OF_ALL_MEMS_FRM_ALL_SRCS',
       'INC_OF_ALL_MEMS_FRM_WAGES'],
      dtype='object')


In [6]:
#Print all the unique values in the coloumn Occupation and their counts
print(df['OCCUPATION_GROUP'].value_counts())

OCCUPATION_GROUP
Data Not Available                       5749711
Wage Labourers                           2942074
Self-employed Entrepreneurs              2519674
Entrepreneurs                            1397505
Small/Marginal Farmers                   1372303
Retired/Aged                             1358716
Support Staff                            1185102
White-collar Professional Employees      1020520
Industrial Workers                        965312
Organised Farmers                         900044
White-collar Clerical Employees           767638
Agricultural Labourers                    651986
Non-industrial Technical Employees        572006
Small Traders/Hawkers                     533138
Business & Salaried Employees             392473
Miscellaneous                             247808
Home-based Workers                        110082
White-collar Employees                    105956
Qualified Self-employed Professionals      86720
Managers/Supervisors                       64194
Sel

In [7]:
#in a seperate df, I want values for Small/Marginal Farmers, Agricultural Labourers, Organised Farmers. 
occupation_groups = ['Small/Marginal Farmers', 'Agricultural Labourers', 'Organised Farmers']
filtered_df = df[df['OCCUPATION_GROUP'].isin(occupation_groups)]


In [ ]:
# aggregate by district annually
# parse year from MONTH (e.g. "Apr 2014")
import numpy as np

df['MONTH_dt'] = pd.to_datetime(df['MONTH'], format='%b %Y', errors='coerce')
df['YEAR'] = df['MONTH_dt'].dt.year

# drop rows without a valid year
df_year = df.dropna(subset=['YEAR']).copy()

# create weighted total income column using household weight
df_year['weighted_tot_inc'] = df_year['TOT_INC'] * df_year['R_HH_WGT_MS']

# group and aggregate
district_year_agg = (
    df_year.groupby(['DISTRICT', 'YEAR'])
    .agg(
        households=('HH_ID', 'size'),
        tot_inc_sum=('TOT_INC', 'sum'),
        tot_inc_mean=('TOT_INC', 'mean'),
        tot_inc_median=('TOT_INC', 'median'),
        wages_sum=('INC_OF_ALL_MEMS_FRM_WAGES', 'sum'),
        weights_sum=('R_HH_WGT_MS', 'sum'),
        weighted_tot_inc_sum=('weighted_tot_inc', 'sum')
    )
    .reset_index()
)

# compute weighted average total income (per district-year)
district_year_agg['weighted_tot_inc_avg'] = (
    district_year_agg['weighted_tot_inc_sum'] / district_year_agg['weights_sum']
).replace([np.inf, -np.inf], np.nan)

# optional: sort for readability
district_year_agg = district_year_agg.sort_values(['DISTRICT', 'YEAR']).reset_index(drop=True)

district_year_agg.head(10)

,DISTRICT,YEAR,households,tot_inc_sum,tot_inc_mean,tot_inc_median,wages_sum,weights_sum,weighted_tot_inc_sum,weighted_tot_inc_avg
0,Adilabad,2014,6814,84174596.0,12353.184033,8500.0,67589201.0,1.241468e+07,1.359374e+11,10949.737094
1,Adilabad,2015,5112,53617524.0,10488.561033,8500.0,43536218.0,8.946415e+06,8.856664e+10,9899.680290
2,Adilabad,2016,6816,74097182.0,10871.065434,9200.0,61796394.0,1.211043e+07,1.263266e+11,10431.219737
3,Adilabad,2017,6786,74985119.0,11049.973327,10300.0,59828427.0,1.157552e+07,1.239762e+11,10710.208109
4,Adilabad,2018,6744,85947393.0,12744.275356,11000.0,59626178.0,1.114200e+07,1.273855e+11,11432.910098
5,Adilabad,2019,6744,94079539.0,13950.109579,12500.0,73720133.0,1.128921e+07,1.349463e+11,11953.565300
6,Adilabad,2020,6808,44702631.0,6566.191392,-99.0,33320053.0,1.143549e+07,5.922064e+10,5178.668845
7,Adilabad,2021,7512,100832199.0,13422.816693,7016.0,71787512.0,1.158019e+07,1.315119e+11,11356.627670
8,Adilabad,2022,7512,189163299.0,25181.482827,16016.0,122393130.0,1.172354e+07,2.373876e+11,20248.798092
9,Adilabad,2023,7544,254537870.0,33740.438759,24016.0,153107972.0,1.191705e+07,3.327065e+11,27918.530953


In [14]:
#in the above district_year_agg df, I want to extract the columns district, year and weighted_tot_inc_avg into a new df
district_year_income = district_year_agg[['DISTRICT', 'YEAR', 'weighted_tot_inc_avg']].copy()
district_year_income.to_csv("district_year_income.csv", index=False)

In [15]:
df = filtered_df

df['MONTH_dt'] = pd.to_datetime(df['MONTH'], format='%b %Y', errors='coerce')
df['YEAR'] = df['MONTH_dt'].dt.year

# drop rows without a valid year
df_year = df.dropna(subset=['YEAR']).copy()

# create weighted total income column using household weight
df_year['weighted_tot_inc'] = df_year['TOT_INC'] * df_year['R_HH_WGT_MS']

# group and aggregate
district_year_agg = (
    df_year.groupby(['DISTRICT', 'YEAR'])
    .agg(
        households=('HH_ID', 'size'),
        tot_inc_sum=('TOT_INC', 'sum'),
        tot_inc_mean=('TOT_INC', 'mean'),
        tot_inc_median=('TOT_INC', 'median'),
        wages_sum=('INC_OF_ALL_MEMS_FRM_WAGES', 'sum'),
        weights_sum=('R_HH_WGT_MS', 'sum'),
        weighted_tot_inc_sum=('weighted_tot_inc', 'sum')
    )
    .reset_index()
)

# compute weighted average total income (per district-year)
district_year_agg['weighted_tot_inc_avg'] = (
    district_year_agg['weighted_tot_inc_sum'] / district_year_agg['weights_sum']
).replace([np.inf, -np.inf], np.nan)

# optional: sort for readability
district_year_agg = district_year_agg.sort_values(['DISTRICT', 'YEAR']).reset_index(drop=True)

district_year_agg.head(10)

C:\Users\Tanishq op\AppData\Local\Temp\ipykernel_18752\2463419673.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['MONTH_dt'] = pd.to_datetime(df['MONTH'], format='%b %Y', errors='coerce')
C:\Users\Tanishq op\AppData\Local\Temp\ipykernel_18752\2463419673.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['YEAR'] = df['MONTH_dt'].dt.year


,DISTRICT,YEAR,households,tot_inc_sum,tot_inc_mean,tot_inc_median,wages_sum,weights_sum,weighted_tot_inc_sum,weighted_tot_inc_avg
0,Adilabad,2014,2453,27809202.0,11336.812882,8500.0,24754648.0,8.488012e+06,9.709916e+10,11439.564181
1,Adilabad,2015,1785,17679856.0,9904.681232,9000.0,16729546.0,5.846166e+06,5.865570e+10,10033.191872
2,Adilabad,2016,2594,27027246.0,10419.138782,9500.0,26003178.0,8.768863e+06,9.236757e+10,10533.586385
3,Adilabad,2017,2570,27734962.0,10791.814008,11000.0,26735568.0,8.777400e+06,9.454074e+10,10770.927572
4,Adilabad,2018,2224,28384556.0,12762.839928,9150.0,23427321.0,7.016101e+06,9.116383e+10,12993.515991
5,Adilabad,2019,1715,27645719.0,16119.952770,12650.0,21772488.0,5.636951e+06,9.095140e+10,16134.855314
6,Adilabad,2020,727,10526941.0,14479.973865,6500.0,9025564.0,2.319178e+06,3.286100e+10,14169.245851
7,Adilabad,2021,1036,27962372.0,26990.706564,12000.0,24618480.0,3.267433e+06,8.779966e+10,26871.140818
8,Adilabad,2022,1597,45826088.0,28695.108328,10016.0,34204720.0,5.295978e+06,1.473115e+11,27815.726211
9,Adilabad,2023,1959,62865236.0,32090.472690,13500.0,36609089.0,6.230288e+06,1.979158e+11,31766.719125


In [16]:
#in the above district_year_agg df, I want to extract the columns district, year and weighted_tot_inc_avg into a new df
district_year_income = district_year_agg[['DISTRICT', 'YEAR', 'weighted_tot_inc_avg']].copy()
district_year_income.to_csv("district_year_income_agri.csv", index=False)